# Support Vector Machine

SVM mencari hyperplane dengan margin terbesar. Posisi hyperplane ditentukan oleh support vector, yaitu titik data yang jaraknya paling dekat ke hyperplane.

Untuk data yang tidak terpisah linear, SVM memakai trik kernel. Kernel menerima dua vektor dan mengembalikan satu bilangan, sehingga hasilnya setara dengan dot-product pada ruang berdimensi lebih tinggi tanpa perlu menghitung transformasinya secara langsung.

Data latihan berisi informasi gizi makanan untuk penderita diabetes. Tugasnya mengklasifikasikan apakah suatu makanan sebaiknya dikonsumsi lebih sering atau lebih jarang.

## Tujuan

- Melatih dan mengevaluasi classifier SVM.
- Menyesuaikan parameter regularisasi dan jenis kernel.
- Menggambarkan hyperplane serta margin.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
)
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.svm import SVC

RANDOM_STATE = 123


## Memuat data


In [ ]:
DATASET_URL = (
    "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/"
    "IBM-ML241EN-SkillsNetwork/labs/datasets/food_items_binary.csv"
)
food_data = pd.read_csv(DATASET_URL)
food_data.head(10)


In [ ]:
feature_columns = list(food_data.iloc[:, :-1].columns)
features = food_data.iloc[:, :-1]
target_encoder = LabelEncoder()
target = pd.Series(target_encoder.fit_transform(food_data.iloc[:, -1]))
features.describe()


Data memuat tujuh belas jenis zat gizi, antara lain kalori, lemak total, protein, gula, dan sodium. Label kelas disandikan menjadi 0 dan 1 agar metrik biner bisa dihitung.


In [ ]:
target.value_counts(normalize=True)


In [ ]:
target.value_counts().plot.bar(color=["red", "green"])
plt.xlabel("Class")
plt.ylabel("Count")
plt.show()


Terdapat dua kelas, yaitu `Less Often` dan `More Often`. Sebarannya tidak seimbang karena sebagian besar makanan masuk kelas `More Often`.


## SVM dengan parameter bawaan


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    features, target, test_size=0.2, stratify=target, random_state=RANDOM_STATE
)

default_model = SVC().fit(X_train, y_train)
predictions = default_model.predict(X_test)


In [ ]:
def evaluate_metrics(actual, predicted):
    precision, recall, f_beta, _ = precision_recall_fscore_support(
        actual, predicted, average="binary"
    )
    return {
        "accuracy": accuracy_score(actual, predicted),
        "recall": recall,
        "precision": precision,
        "f1score": f_beta,
    }


evaluate_metrics(y_test, predictions)


## Regularisasi dan kernel

Parameter `C` mengatur trade-off antara galat latih yang kecil dan batas keputusan yang sederhana. Parameter `kernel` menentukan bentuk transformasi fitur, misalnya `rbf`, `poly`, atau `sigmoid`.


In [ ]:
tuned_model = SVC(C=10, kernel="rbf").fit(X_train, y_train)
evaluate_metrics(y_test, tuned_model.predict(X_test))


## Latihan: mencoba beberapa nilai C dan kernel


In [ ]:
comparison = pd.DataFrame(
    [
        {"C": c_value, "kernel": kernel,
         **evaluate_metrics(
             y_test,
             SVC(C=c_value, kernel=kernel).fit(X_train, y_train).predict(X_test),
         )}
        for c_value in [1, 10, 100]
        for kernel in ["linear", "rbf", "poly"]
    ]
)
comparison


## Grid search untuk C dan kernel


In [ ]:
param_grid = {"C": [1, 10, 100], "kernel": ["poly", "rbf", "sigmoid"]}

search = GridSearchCV(
    SVC(), param_grid=param_grid, scoring="f1", cv=3, n_jobs=-1
).fit(X_train, y_train)
search.best_params_


In [ ]:
best_model = search.best_estimator_
evaluate_metrics(y_test, best_model.predict(X_test))


## Menggambarkan hyperplane dan margin

Visualisasi dibatasi pada dua fitur, yaitu `Calories` dan `Dietary Fiber`, agar bisa digambar dalam bidang dua dimensi. Kelas mayoritas dikurangi jumlahnya supaya gambarnya lebih jelas.


In [ ]:
simplified = food_data[["Calories", "Dietary Fiber", "class"]].head(1000)
minority_label = simplified["class"].value_counts().idxmin()
minority = simplified[simplified["class"] == minority_label]
majority = simplified[simplified["class"] != minority_label].sample(
    n=len(minority), random_state=RANDOM_STATE
)
balanced = pd.concat([minority, majority]).reset_index(drop=True)

X_plot = MinMaxScaler().fit_transform(balanced.iloc[:, :-1].values)
y_plot = LabelEncoder().fit_transform(balanced["class"])
print(f"Dataset resampled shape, X: {X_plot.shape}, y: {y_plot.shape}")


In [ ]:
def plot_decision_boundary(features_2d, labels, model):
    figure, axis = plt.subplots(figsize=(12, 9))
    axis.scatter(features_2d[:, 0], features_2d[:, 1], c=labels, cmap=plt.cm.Paired)

    x_limits = axis.get_xlim()
    y_limits = axis.get_ylim()
    grid_x = np.linspace(x_limits[0], x_limits[1], 30)
    grid_y = np.linspace(y_limits[0], y_limits[1], 30)
    YY, XX = np.meshgrid(grid_y, grid_x)
    grid_points = np.vstack([XX.ravel(), YY.ravel()]).T
    decision = model.decision_function(grid_points).reshape(XX.shape)

    axis.contour(XX, YY, decision, colors="k", levels=[-1, 0, 1], alpha=0.5,
                 linestyles=["--", "-", "--"])
    axis.scatter(model.support_vectors_[:, 0], model.support_vectors_[:, 1],
                 s=100, linewidth=1, facecolors="none", edgecolors="k")
    plt.show()


In [ ]:
linear_model = SVC(C=1000, kernel="linear").fit(X_plot, y_plot)
plot_decision_boundary(X_plot, y_plot, linear_model)


Garis penuh adalah hyperplane, garis putus-putus adalah margin, dan titik yang dilingkari adalah support vector.


In [ ]:
rbf_model = SVC(C=100, kernel="rbf").fit(X_plot, y_plot)
plot_decision_boundary(X_plot, y_plot, rbf_model)


Dengan kernel RBF, batas keputusan menjadi tidak linear pada ruang fitur dua dimensi, meskipun tetap linear pada ruang berdimensi lebih tinggi.


## Latihan: membandingkan pengaruh C dan kernel pada margin


In [ ]:
for c_value, kernel in [(1, "rbf"), (100, "poly"), (1000, "sigmoid")]:
    model = SVC(C=c_value, kernel=kernel).fit(X_plot, y_plot)
    plot_decision_boundary(X_plot, y_plot, model)


## Ringkasan

Nilai `C` besar mempersempit margin agar hampir semua data latih benar, sedangkan `C` kecil memberi margin lebar dengan toleransi kesalahan lebih besar. Kernel RBF paling lentur untuk data yang tidak terpisah linear.
